# Advancement in the Transformer Architecture

### KV Caching

Recall what when we want to predict/generate the next token/word, each heads in the decoding part does the following:

$$Attention(Q,K,V) = \text{Softmax} \left(\frac{Q K^T}{\sqrt{d_k}} + M \right)V$$

This gives us an attention output:

$$O = \begin{bmatrix} o_1 \\ o_1 \\ o_3 \\ \vdots \\ o_n \end{bmatrix}$$

Where each $o_i \in \mathbb{R}^{1 \times d_k}$. Each time we wants to predict the next token, we add another attention output vector $o_{n+1}$ while the rest do not change. As such, recomputing these past attention vectors everytime we want to predict the next token is wasteful. This is where KV-Caching comes in, where we store the Keys and Value vectors from previous tokens in memory. We do not need to cache any Query vectors since all previous Queries pair result in 0 through the masking process. Though effective, its also incredibly memory intensive.

At training time, the Total Memory Access and Total Arithmetic operations for a standard MHA is given by:

$$Memory Access = (\underbrace{bnd}_{\text{XW}} + \underbrace{bhn^2}_{\text{Softmax}} + \underbrace{d^2}_{W^O})$$
$$Operations = (bnd^2)$$

Where $b$ is the batch size, $n$ is the sequence length, and $d$ is the model dimension. As such, our arithmetic intensity (FLOPs/Memory Access) is given by:

$$O\left(\left(\frac{1}{k} + \frac{1}{bn}\right)^{-1}\right)$$

Where $k$ refers to the SRAM capacity, which is the number of tokens that can be stored in the GPU's fast on-chip memory at once. At inference time, the total arithmetic operations are kept the same, but the memory is given by $(bn^2d +nd^2)$. The arithmetic intensity is now given by:

$$O\left(\left(\frac{n}{d} + \frac{1}{b}\right)^{-1}\right)$$

Note that using the MHA architecture, we often do not want shorter sequence length or bigger model in order to increase arithmetic intensity.

<div align="center">
  <img src="../Medias/KVCache.png" width="550">
</div>

### Group Query Attention (GQA)

In multihead attentions, subweights $W_i$ are distinct. In GQA, to reduce the amount of memory KV caching takes, we choose to duplicates certain heads. For example if $h = 4$, we can choose $n_g = 2$ such that there are 2 distinct weights of $W^K$ and $W^V$. We then duplicate these weights until we reach our head counts. Formally:

$$W^K, W^V \in \mathbb{R}^{d_{model} \times (d_k n_g)}$$
$$KW^K = [K'_1,...,K'_{n_g}], VW^V = [V'_1,...,V'_{n_g}]$$

Where $K'_i, V'_i \in \mathbb{R}^{seq \times d_k}$. Using our previous example where $n_g = 2$ and $h = 4$, we then duplicates it in the following ways:

$$K' = [K'_1,K'_1, K'_2, K'_2] = [K'_1,K'_2] 
\begin{bmatrix}
I_{d_k} & I_{d_k} & 0 & 0 \\
0 & 0 & I_{d_k} & I_{d_k}
\end{bmatrix}$$

Note that this is the exact same as Broadcasting. If we choose $n_g = h$, we get our usual Multi-Head Attention, and if we choose $n_g = 1$, then we get Multi-Query Attention.

By reusing some Queries and Keys matrices, we thus cut down on the total memory access while keeping most of the expressiveness of the MHA transformer archetecture. The memory access is now given by $(bnd + bn^2 k +nd^2)$ and arithmetic intensity is given by:

$$O \left( \left(\frac{1}{d} + \frac{n}{dh} + \frac{1}{b} \right)^{-1} \right)$$

This now gives us a way to increase arithmetic intensity without the prior constraints.



### Multi-head Latent Attention (MLA)

<div align="center">
  <img src="../Medias/MLA.png" width="600">
</div>

In multi-head Latent attention, we attempt to compress our Key and Value matrices into a latent space that preserves important aspects of the 2, and then decode that latent space to get back our $W^K$ and $W^V$. Formally, we first compute:

$$L^{KV} = XW^{KV}_{\downarrow}$$

Where $W^{KV} \in \mathbb{R}^{d_{model} \times d_{latent}}$ where $d_{latent} < d_{model}$. We then "up-project" this latent space back via the following:

$$K = L^{KV}W^{K}_{\uparrow}$$
$$V = L^{KV}W^{V}_{\uparrow}$$

While this approach adds additional computes, and thus undermining the purpose of KV cache, we can actually rearrange the archetecture in the following way that minimize both:

$$Q' = XW^Q (W^{K}_{\uparrow})^T$$

We then compute:

$$K' = V' = L^{KV}$$

$$O = Attention(Q', K',  V')$$

We then multiply it back using our new matrix given by:

$$ O' = O W^{V}_{\uparrow} W^O $$

This process can be extented into our usual multi-head approach by splitting up each each $Q', K',$ and $V'$ into different heads, computing their attention, concatinating them together, and then multiply by $W^{V}_{\uparrow} W^O$. We see from this formulation that every head shares the same $L^{KV}$, which is cache and is used by all the attentions. It accomplishes the same thing as KV cache while improving both performance and cost.

### RoPE

The main motivation is that we want to be able to give relative meaning to words that are close apart. Specifically, words that are close together should have similar relation to each other regardless of where they are in absolute term within the sequence. Mathmatically, we want to express it as:

$$\langle f(x,i),f(y,j) \rangle = g(x,y,i-j)$$

Where $x,y$ are the tokens that are being analyzed and $i,j$ are their absolute positions. Here, $f$ is the positional embedding functions, like the sinusoidal position embedding function in the original transformer paper. Most importantly, it says that we want to find a function where only the relative distance between $i$ and $j$ matter. Imagine then that our embedding vector is only 2-d, such that:

$$f(x,n) = R_n x$$

Where $R^n$ is the amount we rotate based on the position of the word. We can check it works:

$$\langle R^i x,R^j y \rangle = (R_ix)^T (R_j y) = (x)^T R_{i-j} (y)$$

RoPE works essentially the same way, where we split up each embedding vectors blocks of 2d vectors, then apply a 2-d rotation matrix on that pair of vector. Every block is assigned its own unique $theta$. The process is summarized as the following:

$$f_{\{q,k\}}(x, m) = R_m W_{\{q,k\}}x $$

Where $x$ is a specific token, $W$ is a weighted matrix of either $W^Q$ or $W^K$, and $m$ is the absolute position of the token within the sequence. $R$ is also defined below as:

$${R} = \begin{pmatrix} 
\cos m\theta_1 & -\sin m\theta_1 & 0 & 0 & \cdots & 0 & 0 \\
\sin m\theta_1 & \cos m\theta_1 & 0 & 0 & \cdots & 0 & 0 \\
0 & 0 & \cos m\theta_2 & -\sin m\theta_2 & \cdots & 0 & 0 \\
0 & 0 & \sin m\theta_2 & \cos m\theta_2 & \cdots & 0 & 0 \\
\vdots & \vdots & \vdots & \vdots & \ddots & \vdots & \vdots \\
0 & 0 & 0 & 0 & \cdots & \cos m\theta_{d/2} & -\sin m\theta_{d/2} \\
0 & 0 & 0 & 0 & \cdots & \sin m\theta_{d/2} & \cos m\theta_{d/2} 
\end{pmatrix}$$

Note that the various $\theta$ is used to capture different kind of informations, and they're not learned but rather set as a hyperparameter. We also see from the equation that RoPE is applied after applying the weighted matrices, but before the attention mechanism, where:

$$Q' = (XW^Q)R_m^T$$
$$K' = (XW^K)R_m^T$$

<div align="center">
  <img src="../Medias/RoPE.png" width="800">
</div>

### Z-loss

In many cases the various softmax functions causes alot of instability when training a model. To reduce instability at the output softmax function (at the end of the architecture), we implement a Z-loss, which is just define as:

$$\mathcal{L}_z = \sum_i \alpha \log^2 (Z(x_i))$$
$$Z(x) = \sum_{r=1}^{|V|} e^{U_r(x)}$$

Where $U_r(x)$ receives a token (at the beginning of the transformer) and return its logit (raw vector value before the output softmax). The Z-loss essentially tries to control the normalizer of the Softmax by forcing it to be near 1, thus preventing it from getting near 0 and causing a spike in the training process.

In our full loss it would look something like this

$$\mathcal{L}_{total} = \frac{1}{B} \sum_{k = 1}^B \Big(\mathcal{L}_k + \alpha \log^2 (Z(x_k)) \Big)$$

### Other changes

Most modern architecture of the transformer puts the Normalization block in front of the attention block rather than behind it, while some opt to do both. Most chose this since choosing a pre-norm design added extra stability and reduced the number of spikes during training.

<div align="center">
  <img src="../Medias/PreNorm.png" width="400">
</div>

Additionally, many models have now swapped out the traditional LayerNorms found in the original transformer paper and have instead used the RMS norm given by:

$$y = \frac{x}{\sqrt{\|x\|^2 + \epsilon}} \cdot \gamma, \text{ Where } \epsilon \text{ is a tiny constant}$$

This approach was choosen to reduce the number of computes and parameters while not sacrificing performance. Non-flops operations like normalization takes up a considerable amount of time due to the memory movement required.

Also a small change in modern transformer architecture design is the removal of the bias term in the FFN layer since it often improve training stability.

Some new activations that models have used is listed:

$$\text{GeLU}(x) = x \Phi(x)$$

Where $\Phi$ is the Cumulative probability of the normal distribution. It looks exactly like the ReLU except its smooth near 0, such that differentiation behaves nicer.

$$\text{ReGLU}(x) = \max (0,xW_1)\otimes xV$$


$$\text{SwiGLU}(x) = xW \cdot sigmoid(xW) \otimes xV$$

These two are what we call Gated Linear Unit (GLU), which has the general form of:

$$\text{GLU}(a, b) = a \otimes \sigma(b)$$
$$a = xW + c_1$$
$$b = xV + c_2$$

Unlike a static activation function, the "gate" depends on the input itself. This allows the model to selectively ignore noise based on the context of the specific token it is processing. Because we added some extra parameters, most models choose to reduce their original $d_{ff}$ be around $2/3$.

Additionally, compare to the original transformer architecture, many models now implement a parralel structure.

<div align="center">
  <img src="../Medias/ParralelLayer.png" width="300">
</div>

As noted above with the Z-loss, the various softmax functions within our transformer can cause significant instability problems. One such place is in the attention mechanism. A way to deal with that however is to take our $Q'$ and $K'$, and pass it through a LayerNorm layer before computing attention.

<div align="center">
  <img src="../Medias/AttentionNorm.png" width="500">
</div>